# AeroVision LK — SAHI Grid Search

**Week 2 | Day 9–10**

**Model:** YOLOv8s baseline weights + SAHI sliced inference

**Goal:** Find optimal slice_size × overlap config to push mAP@0.5 past 55%

**Grid:** 3 slice sizes × 3 overlaps = 9 runs on 548 val images

---

| Config | Slice Sizes | Overlaps | Total Runs |
|--------|------------|----------|------------|
| Grid   | 320, 512, 640 | 0.1, 0.2, 0.3 | 9 |

**Known result:** 512/0.2 = 54.33% mAP@0.5 (notebook 03)

**Baseline:** 44.08% mAP@0.5 (notebook 02)

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import datetime
import yaml
import time
import mlflow
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# ── Project root ──────────────────────────────────────────────────────────────────
def find_project_root():
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if (candidate / 'data' / 'VisDrone_Dataset').exists():
            return candidate
    raise RuntimeError('Cannot find project root. Run from 01_aerovision_lk/ or research/')

PROJECT_ROOT     = find_project_root()
DATASET_ROOT     = PROJECT_ROOT / 'data' / 'VisDrone_Dataset'
DATASET_YAML_ABS = DATASET_ROOT / 'visdrone_abs.yaml'
BASELINE_WEIGHTS = PROJECT_ROOT / 'weights' / 'yolov8s_baseline.pt'
FIGURES_DIR      = PROJECT_ROOT / 'reports' / 'figures'
ANALYSIS_DIR     = PROJECT_ROOT / 'analysis'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

VAL_IMAGES_DIR = DATASET_ROOT / 'VisDrone2019-DET-val' / 'images'
VAL_LABELS_DIR = DATASET_ROOT / 'VisDrone2019-DET-val' / 'labels'

CLASS_NAMES = ['pedestrian', 'people', 'bicycle', 'car', 'van',
               'truck', 'three_wheeler', 'bus', 'motor']
NC = len(CLASS_NAMES)
WEAK_SPOTS = {'bicycle', 'people', 'three_wheeler'}

# ── Grid search config ────────────────────────────────────────────────────────────
SLICE_SIZES    = [320, 512, 640]
OVERLAPS       = [0.1, 0.2, 0.3]
CONF_THRESHOLD = 0.25

# ── Assert prerequisites ──────────────────────────────────────────────────────────
assert BASELINE_WEIGHTS.exists(), f'Missing: {BASELINE_WEIGHTS}'
assert DATASET_YAML_ABS.exists(), f'Missing: {DATASET_YAML_ABS}'

# ── Load baseline metrics ─────────────────────────────────────────────────────────
baseline_csv = ANALYSIS_DIR / 'baseline_metrics.csv'
assert baseline_csv.exists(), f'Missing: {baseline_csv}'
baseline_row     = pd.read_csv(baseline_csv).iloc[-1]
BASELINE_MAP50   = float(baseline_row['mAP50'])
BASELINE_MAP5095 = float(baseline_row['mAP50_95'])

# ── Load notebook 03 reference ────────────────────────────────────────────────────
sahi_csv = ANALYSIS_DIR / 'sahi_metrics.csv'
assert sahi_csv.exists(), f'Missing: {sahi_csv}'
sahi_ref_row   = pd.read_csv(sahi_csv).iloc[-1]
SAHI_REF_MAP50 = float(sahi_ref_row['mAP50'])

n_val    = len(list(VAL_IMAGES_DIR.glob('*.jpg')))
n_combos = len(SLICE_SIZES) * len(OVERLAPS)

print(f'Project root       : {PROJECT_ROOT}')
print(f'Baseline weights   : {BASELINE_WEIGHTS.name}  ({BASELINE_WEIGHTS.stat().st_size/1024/1024:.1f} MB)')
print(f'Val images         : {n_val}')
print(f'Baseline mAP@0.5   : {BASELINE_MAP50*100:.2f}%')
print(f'NB03 ref mAP@0.5   : {SAHI_REF_MAP50*100:.2f}%  (512/0.2)')
print(f'Target mAP@0.5     : 55%+')
print(f'Grid configurations: {n_combos}  ({SLICE_SIZES} x {OVERLAPS})')
print(f'Est. runtime       : ~{n_combos * 2.5:.0f} min')

---
## 1. Shared Helpers

In [ ]:
def load_label(label_path: Path):
    """Return (boxes_norm [N,4], class_ids [N]) from a YOLO label file."""
    rows, cids = [], []
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) == 5:
            cids.append(int(parts[0]))
            rows.append([float(x) for x in parts[1:]])
    boxes = np.array(rows, dtype=np.float32) if rows else np.zeros((0, 4), dtype=np.float32)
    return boxes, cids


def yolo_to_xyxy(boxes_norm, img_w, img_h):
    """Convert YOLO normalised [cx,cy,w,h] -> pixel [x1,y1,x2,y2]."""
    if len(boxes_norm) == 0:
        return np.zeros((0, 4), dtype=np.float32)
    cx, cy, bw, bh = boxes_norm[:,0], boxes_norm[:,1], boxes_norm[:,2], boxes_norm[:,3]
    x1 = (cx - bw/2) * img_w
    y1 = (cy - bh/2) * img_h
    x2 = (cx + bw/2) * img_w
    y2 = (cy + bh/2) * img_h
    return np.stack([x1, y1, x2, y2], axis=1).astype(np.float32)


def iou_matrix(gt_xyxy, pred_xyxy):
    """Vectorized IoU matrix [N_gt, N_pred]. No external dependencies."""
    if len(gt_xyxy) == 0 or len(pred_xyxy) == 0:
        return np.zeros((len(gt_xyxy), len(pred_xyxy)), dtype=np.float32)
    gt   = gt_xyxy[:, None, :]
    pred = pred_xyxy[None, :, :]
    ix1  = np.maximum(gt[...,0], pred[...,0])
    iy1  = np.maximum(gt[...,1], pred[...,1])
    ix2  = np.minimum(gt[...,2], pred[...,2])
    iy2  = np.minimum(gt[...,3], pred[...,3])
    inter = np.maximum(0.0, ix2-ix1) * np.maximum(0.0, iy2-iy1)
    area_gt   = (gt_xyxy[:,2]-gt_xyxy[:,0]) * (gt_xyxy[:,3]-gt_xyxy[:,1])
    area_pred = (pred_xyxy[:,2]-pred_xyxy[:,0]) * (pred_xyxy[:,3]-pred_xyxy[:,1])
    union = area_gt[:,None] + area_pred[None,:] - inter
    return inter / np.maximum(union, 1e-6)


def compute_ap(recalls, precisions):
    """101-point interpolated AP (COCO-style)."""
    r = np.concatenate(([0.0], recalls,    [1.0]))
    p = np.concatenate(([1.0], precisions, [0.0]))
    for i in range(len(p)-2, -1, -1):
        p[i] = max(p[i], p[i+1])
    x  = np.linspace(0, 1, 101)
    ap = float(np.trapezoid(np.interp(x, r, p), x))
    return ap


def compute_map(all_preds, all_gts, iou_threshold=0.5):
    """
    Compute mAP across all classes.
    all_preds: list of dicts {class_id, conf, img_id, x1,y1,x2,y2}
    all_gts:   list of dicts {class_id, img_id, gt_idx, x1,y1,x2,y2}
    Returns: (mAP float, per_class_ap dict {cid: float})
    """
    per_class_ap = {}
    for cid in range(NC):
        preds = sorted([p for p in all_preds if p['class_id'] == cid], key=lambda x: -x['conf'])
        gts   = [g for g in all_gts if g['class_id'] == cid]
        if not gts:
            continue
        matched = set()
        tp = np.zeros(len(preds))
        fp = np.zeros(len(preds))
        for i, pred in enumerate(preds):
            img_gts = [g for g in gts if g['img_id'] == pred['img_id']]
            if not img_gts:
                fp[i] = 1; continue
            pb  = np.array([[pred['x1'], pred['y1'], pred['x2'], pred['y2']]], dtype=np.float32)
            gb  = np.array([[g['x1'],g['y1'],g['x2'],g['y2']] for g in img_gts], dtype=np.float32)
            ious = iou_matrix(gb, pb)[:,0]
            best = int(np.argmax(ious))
            key  = (pred['img_id'], img_gts[best]['gt_idx'])
            if ious[best] >= iou_threshold and key not in matched:
                tp[i] = 1; matched.add(key)
            else:
                fp[i] = 1
        cum_tp = np.cumsum(tp)
        cum_fp = np.cumsum(fp)
        rec  = cum_tp / (len(gts) + 1e-6)
        prec = cum_tp / (cum_tp + cum_fp + 1e-6)
        per_class_ap[cid] = compute_ap(rec, prec)
    mAP = float(np.mean(list(per_class_ap.values()))) if per_class_ap else 0.0
    return mAP, per_class_ap


def sahi_to_boxes(sahi_result):
    """Extract (boxes_xyxy [N,4], class_ids [N], confs [N]) from SAHI result."""
    boxes, cids, confs = [], [], []
    for obj in sahi_result.object_prediction_list:
        b = obj.bbox
        boxes.append([b.minx, b.miny, b.maxx, b.maxy])
        cids.append(int(obj.category.id))
        confs.append(float(obj.score.value))
    boxes = np.array(boxes, dtype=np.float32) if boxes else np.zeros((0,4), dtype=np.float32)
    return boxes, cids, confs


print('Helpers defined.')

---
## 2. Load Detection Model

In [ ]:
detection_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics',
    model_path=str(BASELINE_WEIGHTS),
    confidence_threshold=CONF_THRESHOLD,
    device='cuda:0',
)
print(f'SAHI model loaded      : confidence_threshold={CONF_THRESHOLD}, device=cuda:0')
print(f'Ready for grid search  : {len(SLICE_SIZES) * len(OVERLAPS)} configurations')

---
## 3. Grid Search — 9 Configurations

Expected runtime: ~23 min total (~2.5 min per config on RTX 3050)

In [ ]:
# ── Load ground truth ONCE ─────────────────────────────────────────────────────────
val_images = sorted(VAL_IMAGES_DIR.glob('*.jpg'))
n_images   = len(val_images)

print('Loading ground truth for all val images ...')
all_gts = []

for img_path in val_images:
    label_path = VAL_LABELS_DIR / (img_path.stem + '.txt')
    if not label_path.exists():
        continue
    img      = cv2.imread(str(img_path))
    img_h, img_w = img.shape[:2]
    img_id   = img_path.stem

    gt_norm, gt_cids = load_label(label_path)
    gt_xyxy          = yolo_to_xyxy(gt_norm, img_w, img_h)
    for gt_idx, (box, cid) in enumerate(zip(gt_xyxy, gt_cids)):
        all_gts.append({'class_id': cid, 'img_id': img_id, 'gt_idx': gt_idx,
                        'x1': float(box[0]), 'y1': float(box[1]),
                        'x2': float(box[2]), 'y2': float(box[3])})

print(f'  GT loaded: {len(all_gts):,} boxes across {n_images} images')
print()

# ── Grid search ──────────────────────────────────────────────────────────────────
grid_results = []
run_idx      = 0
total_runs   = len(SLICE_SIZES) * len(OVERLAPS)
grid_t_start = datetime.datetime.now()

for slice_size in SLICE_SIZES:
    for overlap in OVERLAPS:
        run_idx += 1
        detection_model.confidence_threshold = CONF_THRESHOLD

        print('=' * 62)
        print(f'  [{run_idx}/{total_runs}]  slice={slice_size}  overlap={overlap}  conf={CONF_THRESHOLD}')
        print('=' * 62)

        all_preds = []
        t_start   = time.time()

        for img_idx, img_path in enumerate(val_images):
            label_path = VAL_LABELS_DIR / (img_path.stem + '.txt')
            if not label_path.exists():
                continue
            img_id = img_path.stem

            sahi_res = get_sliced_prediction(
                str(img_path), detection_model,
                slice_height=slice_size, slice_width=slice_size,
                overlap_height_ratio=overlap, overlap_width_ratio=overlap,
                verbose=False,
            )
            pred_boxes, pred_cids, pred_confs = sahi_to_boxes(sahi_res)
            for box, cid, conf in zip(pred_boxes, pred_cids, pred_confs):
                all_preds.append({'class_id': int(cid), 'conf': float(conf),
                                  'img_id': img_id,
                                  'x1': float(box[0]), 'y1': float(box[1]),
                                  'x2': float(box[2]), 'y2': float(box[3])})

            if (img_idx + 1) % 100 == 0:
                elapsed = time.time() - t_start
                rate    = (img_idx + 1) / max(elapsed, 1e-6)
                eta     = (n_images - img_idx - 1) / max(rate, 1e-6)
                print(f'    {img_idx+1}/{n_images}  ({rate:.1f} img/s, ETA {eta/60:.1f} min)')

        t_elapsed  = time.time() - t_start
        ms_per_img = (t_elapsed / n_images) * 1000

        # ── Compute mAP ──
        map50, ap50_per_class = compute_map(all_preds, all_gts, iou_threshold=0.5)

        iou_thresholds = np.arange(0.5, 1.0, 0.05)
        map5095_list   = [map50]
        for thresh in iou_thresholds[1:]:
            m, _ = compute_map(all_preds, all_gts, iou_threshold=round(float(thresh), 2))
            map5095_list.append(m)
        map5095 = float(np.mean(map5095_list))

        result = {
            'slice_size':     slice_size,
            'overlap':        overlap,
            'conf_threshold': CONF_THRESHOLD,
            'mAP50':          round(map50, 4),
            'mAP50_95':       round(map5095, 4),
            'ms_per_img':     round(ms_per_img, 1),
            'n_preds':        len(all_preds),
            'timestamp':      datetime.datetime.now().isoformat(timespec='seconds'),
        }
        for cid, name in enumerate(CLASS_NAMES):
            result[f'AP50_{name}'] = round(ap50_per_class.get(cid, 0.0), 4)
        grid_results.append(result)

        delta_base = (map50 - BASELINE_MAP50) * 100
        print(f'  -> mAP@0.5={map50*100:.2f}%  mAP@0.5:0.95={map5095*100:.2f}%  '
              f'({delta_base:+.2f}pp vs baseline)  {ms_per_img:.0f} ms/img  '
              f'{len(all_preds):,} preds  [{t_elapsed/60:.1f} min]')
        print()

grid_t_elapsed = (datetime.datetime.now() - grid_t_start).total_seconds()
print('=' * 62)
print(f'  Grid search complete: {total_runs} configs in {grid_t_elapsed/60:.1f} min')
print('=' * 62)

---
## 4. Results Table

In [ ]:
df = pd.DataFrame(grid_results).sort_values('mAP50', ascending=False).reset_index(drop=True)

best_idx  = 0
best_row  = df.iloc[best_idx]
best_map  = best_row['mAP50']
best_cfg  = f'{int(best_row["slice_size"])}/{best_row["overlap"]}'

print('=' * 80)
print('  SAHI Grid Search Results \u2014 Sorted by mAP@0.5')
print('=' * 80)
print(f'  {"#":<4} {"Slice":>6} {"Overlap":>8} {"mAP@0.5":>9} {"mAP@.5:.95":>11} '
      f'{"ms/img":>8} {"Preds":>8} {"vs Base":>9}')
print('  ' + '-' * 74)

for i, row in df.iterrows():
    delta = (row['mAP50'] - BASELINE_MAP50) * 100
    marker = '  <-- BEST' if i == best_idx else ''
    print(f'  {i+1:<4} {int(row["slice_size"]):>6} {row["overlap"]:>8.1f} '
          f'{row["mAP50"]*100:>8.2f}% {row["mAP50_95"]*100:>10.2f}% '
          f'{row["ms_per_img"]:>7.0f}ms {row["n_preds"]:>8,} {delta:>+8.2f}pp{marker}')

print('=' * 80)
print()
print(f'  Best config        : {best_cfg}  ({best_map*100:.2f}% mAP@0.5)')
print(f'  Delta vs baseline  : {(best_map - BASELINE_MAP50)*100:+.2f}pp')
print(f'  Delta vs NB03 ref  : {(best_map - SAHI_REF_MAP50)*100:+.2f}pp')
print(f'  Target 55%+ met    : {"YES" if best_map >= 0.55 else "NO"}  ({best_map*100:.2f}%)')

---
## 5. Heatmap — Slice Size vs Overlap vs mAP@0.5

In [ ]:
pivot = df.pivot_table(index='slice_size', columns='overlap', values='mAP50', aggfunc='first')
pivot = pivot.sort_index(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values * 100, cmap='YlGn', aspect='auto',
               vmin=pivot.values.min()*100 - 1, vmax=pivot.values.max()*100 + 1)

# Annotate cells
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val   = pivot.values[i, j] * 100
        delta = val - BASELINE_MAP50 * 100
        text  = f'{val:.2f}%\n({delta:+.1f}pp)'
        color = 'white' if val > (pivot.values.max()*100 - 2) else 'black'
        ax.text(j, i, text, ha='center', va='center', fontsize=10,
                fontweight='bold' if val == pivot.values.max()*100 else 'normal',
                color=color)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{o:.1f}' for o in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([str(s) for s in pivot.index])
ax.set_xlabel('Overlap Ratio', fontsize=11)
ax.set_ylabel('Slice Size (px)', fontsize=11)
ax.set_title(f'SAHI Grid Search \u2014 mAP@0.5 (baseline={BASELINE_MAP50*100:.1f}%)', fontsize=12)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('mAP@0.5 (%)')

plt.tight_layout()
out_path = FIGURES_DIR / 'sahi_grid_heatmap.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved -> {out_path}')
plt.show()

---
## 6. Save Results & Log to MLflow

In [ ]:
# ── Save CSV ────────────────────────────────────────────────────────────────────
csv_path = ANALYSIS_DIR / 'sahi_grid_search.csv'
df.to_csv(csv_path, index=False)
print(f'Saved -> {csv_path}  ({len(df)} rows)')

# ── MLflow — one run per config ─────────────────────────────────────────────────
mlflow.set_tracking_uri((PROJECT_ROOT / 'mlruns').as_uri())
mlflow.set_experiment('aerovision-lk')

run_ids = []
for _, row in df.iterrows():
    run_name = f'grid-{int(row["slice_size"])}-{str(row["overlap"]).replace(".", "")}'
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params({
            'model':          'yolov8s',
            'sahi':           True,
            'slice_size':     int(row['slice_size']),
            'overlap_ratio':  row['overlap'],
            'conf_threshold': row['conf_threshold'],
            'dataset':        'VisDrone2019-DET-9class',
            'experiment':     'grid_search',
        })
        mlflow.log_metric('val/mAP50',    row['mAP50'])
        mlflow.log_metric('val/mAP50-95', row['mAP50_95'])
        mlflow.log_metric('ms_per_img',   row['ms_per_img'])
        mlflow.log_metric('n_preds',      row['n_preds'])
        for name in CLASS_NAMES:
            col = f'AP50_{name}'
            if col in row.index:
                mlflow.log_metric(f'AP50_{name}', row[col])
        # Attach CSV and heatmap to best run only
        if row['mAP50'] == best_map:
            mlflow.log_artifact(str(csv_path), artifact_path='metrics')
            mlflow.log_artifact(str(FIGURES_DIR / 'sahi_grid_heatmap.png'), artifact_path='figures')
        run_ids.append(run.info.run_id)

print(f'Logged {len(run_ids)} MLflow runs to experiment aerovision-lk')
print(f'Best run ID: {run_ids[0]}')

In [ ]:
best = df.iloc[0]
second = df.iloc[1] if len(df) > 1 else None

print('=' * 62)
print('  BEST CONFIGURATION RECOMMENDATION')
print('=' * 62)
print(f'  Slice size   : {int(best["slice_size"])}px')
print(f'  Overlap      : {best["overlap"]}')
print(f'  mAP@0.5      : {best["mAP50"]*100:.2f}%')
print(f'  mAP@0.5:0.95 : {best["mAP50_95"]*100:.2f}%')
print(f'  Latency      : {best["ms_per_img"]:.0f} ms/img')
print(f'  Predictions  : {best["n_preds"]:,}')
print()
print(f'  vs Baseline  : {(best["mAP50"]-BASELINE_MAP50)*100:+.2f}pp')
print(f'  vs NB03 ref  : {(best["mAP50"]-SAHI_REF_MAP50)*100:+.2f}pp')
print()

if best['mAP50'] >= 0.55:
    print('  TARGET 55%+ : MET')
else:
    gap = (0.55 - best['mAP50']) * 100
    print(f'  TARGET 55%+ : NOT MET (gap: {gap:.2f}pp)')

if second is not None:
    diff_map = (best['mAP50'] - second['mAP50']) * 100
    diff_ms  = best['ms_per_img'] - second['ms_per_img']
    print()
    print(f'  Runner-up    : {int(second["slice_size"])}/{second["overlap"]} '
          f'({second["mAP50"]*100:.2f}%, {second["ms_per_img"]:.0f} ms/img)')
    print(f'  Margin       : {diff_map:+.2f}pp mAP, {diff_ms:+.0f} ms/img latency')

    if diff_map < 0.5 and abs(diff_ms) > 50:
        print(f'  Note: runner-up is within 0.5pp and {abs(diff_ms):.0f}ms different \u2014 consider it.')

print()
print('  Next steps:')
print('    1. Update analysis/sahi_metrics.csv with best config')
print('    2. Use best config for failure analysis (notebook 05)')
print('    3. ONNX export + INT8 quantization (Week 3)')
print('=' * 62)